# Colab: train a supported translation pair
Run cells in order. Select a GPU runtime if available; no runtime duration or GPU availability is guaranteed. Default: English to Twi using [Helsinki-NLP/opus-mt-en-tw](https://huggingface.co/Helsinki-NLP/opus-mt-en-tw). This is not a Twi/French model.
Commit and push this working tree to your GitHub first, or upload the complete updated project to `/content/TW-FR-MT`. Uploading only this notebook does not upload package changes. The original GitHub repository will not contain local changes until pushed. Never embed credentials in this notebook.

In [ ]:
from pathlib import Path
import subprocess
import os
from urllib.parse import urlparse

REPO_URL = 'https://github.com/gyasifred/TW-FR-MT.git'  # Change to your pushed repository.
BRANCH = ''  # Optional branch; existing checkouts are never switched or pulled.
REPO_DIR = Path('/content/TW-FR-MT')
parsed = urlparse(REPO_URL)
if parsed.scheme != 'https' or parsed.hostname != 'github.com' or parsed.username or parsed.password or parsed.query or parsed.fragment:
    raise ValueError('Use a credential-free HTTPS GitHub repository URL.')
if not REPO_DIR.exists():
    command = ['git', 'clone']
    if BRANCH:
        command += ['--branch', BRANCH, '--single-branch']
    subprocess.run(command + ['--', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Using existing checkout/upload without overwriting it:', REPO_DIR)
if (REPO_DIR / '.git').exists():
    if BRANCH:
        actual = subprocess.check_output(['git', '-C', str(REPO_DIR), 'branch', '--show-current'], text=True).strip()
        if actual != BRANCH:
            raise RuntimeError(f'Existing branch {actual!r} differs from {BRANCH!r}; resolve manually.')
    subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], check=True)
if not (REPO_DIR / 'pyproject.toml').is_file():
    raise RuntimeError('Updated package missing. Push the working tree first or upload the complete updated project.')
os.chdir(REPO_DIR)


In [ ]:
%pip install -e '.[train,app]'
# The project dependency pin is transformers==4.57.1.
from importlib.metadata import version
assert version('transformers') == '4.57.1', 'Check pyproject.toml and restart the runtime after installing.'


## Authorize Drive
Training data: [data](https://drive.google.com/drive/folders/139FuEvqxcZzVglh0qNBIIDaXusjpgFuh?usp=drive_link), mounted at `/content/drive/MyDrive/data`. Model outputs: [MT](https://drive.google.com/drive/folders/13irIvPsqnryP_NJ5y6PneFKrQDJs5Qm6?usp=drive_link), mounted at `/content/drive/MyDrive/MT`. The configuration uses the exact filenames from your saved inventory. Run `00_drive_inventory.ipynb` to recheck availability. Mounting grants notebook code access to Drive. Corpus contents and alignment still require validation; no public download helper is used.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
DATA_FOLDER_ID = '139FuEvqxcZzVglh0qNBIIDaXusjpgFuh'
MODEL_FOLDER_ID = '13irIvPsqnryP_NJ5y6PneFKrQDJs5Qm6'
DATA_ROOT = Path('/content/drive/MyDrive/data')
DRIVE_ROOT = Path('/content/drive/MyDrive/MT')
BASE_MODEL = 'Helsinki-NLP/opus-mt-en-tw'
SOURCE_LANG, TARGET_LANG = 'en', 'tw'
DATA_FILES = {
    'train': {'en': DATA_ROOT / 'train_en', 'tw': DATA_ROOT / 'train_twi', 'fr': DATA_ROOT / 'train_fr'},
    'validation': {'en': DATA_ROOT / 'val_en', 'tw': DATA_ROOT / 'val_tw', 'fr': DATA_ROOT / 'val_fr'},
    'test': {'en': DATA_ROOT / 'test_en', 'tw': DATA_ROOT / 'test_twi', 'fr': DATA_ROOT / 'test_fr'},
}
INPUT_FILES = {split: (files[SOURCE_LANG], files[TARGET_LANG]) for split, files in DATA_FILES.items()}
for folder in (DATA_ROOT, DRIVE_ROOT):
    if not folder.is_dir():
        raise FileNotFoundError(f'Mounted Drive folder missing: {folder}')
for split, paths in INPUT_FILES.items():
    for path in paths:
        if not path.is_file():
            raise FileNotFoundError(f'{split} input missing: {path}')
RUN_NAME = 'en-tw-run-001'  # Use a new name for each fresh experiment.
if Path(RUN_NAME).name != RUN_NAME or RUN_NAME in ('', '.', '..'):
    raise ValueError('RUN_NAME must be one directory name.')
RUN_DIR = DRIVE_ROOT / 'runs' / RUN_NAME
SPLIT_DIR = Path('/content/twfr-splits') / RUN_NAME
LOCAL_DATA_DIR = Path('/content/twfr-inputs') / RUN_NAME
RESUME_FROM_CHECKPOINT = None  # Set a real checkpoint directory only when resuming.
if BASE_MODEL == 'Helsinki-NLP/opus-mt-en-tw' and (SOURCE_LANG, TARGET_LANG) != ('en', 'tw'):
    raise ValueError('This base model supports en -> tw only. Select a compatible model for another pair.')
if RESUME_FROM_CHECKPOINT is None and RUN_DIR.exists() and any(RUN_DIR.iterdir()):
    raise FileExistsError('Run already contains files. Choose a new RUN_NAME or an explicit resume checkpoint.')
if RESUME_FROM_CHECKPOINT is not None and not Path(RESUME_FROM_CHECKPOINT).is_dir():
    raise FileNotFoundError(RESUME_FROM_CHECKPOINT)


## Validate your existing splits
Copy the selected English/Twi training, validation and test files from Drive to local Colab storage. Preserve their split membership; do not split the training file again or merge held-out data into training. Strict parsing rejects blank rows, empty files and unequal lengths. The split audit rejects duplicate exact pairs and cross-split source overlap rather than silently repairing data. Inspect sentence alignment and language direction before training; matching lengths alone do not establish alignment. Only train and validation enter training; test is reserved for final frozen evaluation.
For another pair, select compatible data and a compatible base model together. Changing language labels does not establish model support. Verify tokenizer language codes and any required prefix; do not invent Twi/French tokenizer tokens.

In [ ]:
import json
import shutil
from twfr_mt.data import corpus_lineage, read_parallel, validate_splits

if SPLIT_DIR.exists() or LOCAL_DATA_DIR.exists():
    raise FileExistsError('Inspect existing local inputs/splits before rerunning; use a fresh runtime for resume.')
LOCAL_DATA_DIR.mkdir(parents=True)
inputs = {}
for split, paths in INPUT_FILES.items():
    inputs[split] = corpus_lineage(*paths)
    for side, path in zip(('source', 'target'), paths):
        shutil.copy2(path, LOCAL_DATA_DIR / f'{split}.{side}.txt')
    local_paths = [LOCAL_DATA_DIR / f'{split}.{side}.txt' for side in ('source', 'target')]
    copied = corpus_lineage(*local_paths)
    if inputs[split] != corpus_lineage(*paths) or any(
        inputs[split][side]['sha256'] != copied[side]['sha256'] for side in ('source', 'target')
    ):
        raise ValueError(f'{split} inputs changed during copying.')
splits = {split: read_parallel(LOCAL_DATA_DIR / f'{split}.source.txt',
                               LOCAL_DATA_DIR / f'{split}.target.txt') for split in INPUT_FILES}
audit = validate_splits(splits)
shutil.copytree(LOCAL_DATA_DIR, SPLIT_DIR)
manifest = {'schema_version': 1, 'policy': 'Preserve supplied split membership; no resplitting',
            'source_lang': SOURCE_LANG, 'target_lang': TARGET_LANG,
            'inputs': inputs, 'counts': audit['counts']}
(SPLIT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('Split counts:', audit['counts'])
saved_splits = DRIVE_ROOT / 'prepared_data' / RUN_NAME
if saved_splits.exists():
    local_files = {p.name: p.read_bytes() for p in SPLIT_DIR.iterdir() if p.is_file()}
    saved_files = {p.name: p.read_bytes() for p in saved_splits.iterdir() if p.is_file()}
    if local_files != saved_files:
        raise ValueError('Saved split provenance differs. Do not resume with changed data.')
else:
    shutil.copytree(SPLIT_DIR, saved_splits)
print('Manifest and validated splits preserved at:', saved_splits)


## Configure training
The next cell uses the package's training configuration. Check paths and settings before running. Checkpoints and the final export go to this Drive run directory; reading split files locally reduces Drive I/O. Drive checkpoint writes can still be slow or interrupted. Resume requires the same model, data split, and training settings as the saved run.

In [ ]:
from twfr_mt.training import TrainingConfig, run_training
# Model revision can be pinned to a Hub commit for reproducibility.
MODEL_REVISION = 'main'
EPOCHS = 3
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-5
SOURCE_TOKENIZER_LANG = None  # Marian default requires no invented language tokens.
TARGET_TOKENIZER_LANG = None
SOURCE_PREFIX = ''


In [ ]:
config = TrainingConfig(
    base_model=BASE_MODEL, revision=MODEL_REVISION,
    source_lang=SOURCE_LANG, target_lang=TARGET_LANG,
    train_source=str(SPLIT_DIR / 'train.source.txt'),
    train_target=str(SPLIT_DIR / 'train.target.txt'),
    validation_source=str(SPLIT_DIR / 'validation.source.txt'),
    validation_target=str(SPLIT_DIR / 'validation.target.txt'),
    output_dir=str(RUN_DIR), resume_from_checkpoint=RESUME_FROM_CHECKPOINT,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, learning_rate=LEARNING_RATE,
    max_source_length=128, max_target_length=128, seed=42,
    source_tokenizer_lang=SOURCE_TOKENIZER_LANG, target_tokenizer_lang=TARGET_TOKENIZER_LANG,
    source_prefix=SOURCE_PREFIX, save_total_limit=2, device='auto',
)
print(config)


In [ ]:
# This starts training. Do not use test scores for hyperparameter or checkpoint selection.
training_result = run_training(config)
print(training_result)
print('Inspect saved artifacts under:', RUN_DIR)


## After training
Verify a complete model/tokenizer export in `RUN_DIR/final` before opening `02_colab_translate.ipynb`. Point its explicit checkpoint setting at this relative Drive path. Preserve the split manifest and run configuration with the experiment. After freezing the model, generate predictions on the untouched test source and evaluate once; do not feed the test pair into Trainer. No training or Colab execution is claimed by this notebook's authors. See `docs/COLAB.md` and `docs/DEPLOYMENT.md` for persistence and hosting limits.